### FCC Processing Workflow for the state of California (CA)

In [1]:
### Importing required libraries
import os
import pandas as pd

In [2]:
### Reading all the CA CSV files into a DataFrame
path = '../data/CA/'
ca_data = pd.DataFrame()
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(path, file))
        ca_data = pd.concat([ca_data, df], ignore_index=True)
ca_data.head()

,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id
0,18735928,131173,AUDEAMUS,1332781653,50,200,100,1,X,CA,60190082007019,8829a95b55fffff
1,7494776,130335,Fidium Fiber,1350860893,50,2000,2000,1,R,CA,60610209083005,882832ab0dfffff
2,7494776,130335,Fidium Fiber,1350864715,50,2000,2000,1,R,CA,60610211091015,88283214e3fffff
3,7494776,130335,Fidium Fiber,1350882146,50,2000,2000,1,R,CA,60610210431018,882832ab39fffff
4,7494776,130335,Fidium Fiber,1350884836,50,2000,2000,1,R,CA,60610210341006,882832ab35fffff


In [3]:
### Running basic data checks
print(ca_data.info())

### Checking for missing values
print(ca_data.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 19402734 entries, 0 to 19402733
Data columns (total 12 columns):
 #   Column                         Dtype
---  ------                         -----
 0   frn                            int64
 1   provider_id                    int64
 2   brand_name                     str  
 3   location_id                    int64
 4   technology                     int64
 5   max_advertised_download_speed  int64
 6   max_advertised_upload_speed    int64
 7   low_latency                    int64
 8   business_residential_code      str  
 9   state_usps                     str  
 10  block_geoid                    int64
 11  h3_res8_id                     str  
dtypes: int64(8), str(4)
memory usage: 1.7 GB
None
frn                              0
provider_id                      0
brand_name                       0
location_id                      0
technology                       0
max_advertised_download_speed    0
max_advertised_upload_speed      0
low_latency 

In [4]:
### Checking unique values for categorical columns
print(ca_data['technology'].unique())
print(ca_data['provider_id'].unique().shape)

[50 40 10]
(90,)


In [5]:
### Reading providers master sheet
providers_master_data = pd.read_csv('../output/final_master_set.csv')
providers_master_data.head()

,Provider ID,FRN,holding_company
0,240058,12609,Mid-Hudson Cablevision
1,240068,12781,"Neu Ventures, Inc."
2,131087,13722,"Rainbow Telecommunications Association, Inc."
3,240058,14936,Mid-Hudson Cablevision
4,131060,17640,"TPC ACC Acquisition, LLC"


In [6]:
### Renaming columns for merging
providers_master_data.rename(columns={'Provider ID': 'provider_id', 'FRN':'frn'}, inplace=True)

In [7]:
### Merging CA data with providers master data to get provider names
merged_data = pd.merge(ca_data, providers_master_data, on=['provider_id', 'frn'], how='left')

In [8]:
### Printing out both the shape of the merged data and the shape of CA data file to check if there are any discrepancies
print("CA Data File Shape:", ca_data.shape)
print("Merged Data Shape:", merged_data.shape)

CA Data File Shape: (19402734, 12)
Merged Data Shape: (19402734, 13)


In [9]:
### Finding out the counts of Residential and Business customers in the merged data
merged_data['business_residential_code'].value_counts()

business_residential_code
X    11406006
R     7513255
B      483473
Name: count, dtype: int64

### For the purpose of this project we will consider **Residential Addresses Only**

In [10]:
### Filtering out non residential buildings from the merged data
residential_data = merged_data[merged_data['business_residential_code'] == 'R']

In [11]:
### The CB code is a 15-digit code, Adding a padleft to make the necessary leading zeros to the CB code
residential_data['CB_Code'] = residential_data['block_geoid'].apply(lambda x: str(x).zfill(15))
print(residential_data['CB_Code'].value_counts())
residential_data.head()

CB_Code
060650441031018    2064
060379203391006    1382
060650430013000    1367
060710019081028    1259
060710019063000    1249
                   ... 
060710104102004       1
060379009011007       1
060379001021182       1
060830019051036       1
060830020062034       1
Name: count, Length: 212659, dtype: int64


,frn,provider_id,brand_name,location_id,technology,max_advertised_download_speed,max_advertised_upload_speed,low_latency,business_residential_code,state_usps,block_geoid,h3_res8_id,holding_company,CB_Code
1,7494776,130335,Fidium Fiber,1350860893,50,2000,2000,1,R,CA,60610209083005,882832ab0dfffff,"Fidium Holdings, LLC",060610209083005
2,7494776,130335,Fidium Fiber,1350864715,50,2000,2000,1,R,CA,60610211091015,88283214e3fffff,"Fidium Holdings, LLC",060610211091015
3,7494776,130335,Fidium Fiber,1350882146,50,2000,2000,1,R,CA,60610210431018,882832ab39fffff,"Fidium Holdings, LLC",060610210431018
4,7494776,130335,Fidium Fiber,1350884836,50,2000,2000,1,R,CA,60610210341006,882832ab35fffff,"Fidium Holdings, LLC",060610210341006
5,7494776,130335,Fidium Fiber,1350888891,50,2000,2000,1,R,CA,60610224001013,88283214c7fffff,"Fidium Holdings, LLC",060610224001013


In [12]:
### Dropping unnecessary columns from the residential data
columns_to_drop = ['block_geoid', 'business_residential_code', 'brand_name', 'frn', 'low_latency', 'h3_res8_id', 'location_id']
residential_data.drop(columns=columns_to_drop, inplace=True)


In [13]:
### Taking the maximum tech per provider at each CB code level
residential_data['max_tech'] = residential_data.groupby(['CB_Code', 'holding_company'])['technology'].transform('max')

### Descending sort on max_tech for each  CB and provider combination to get the best technology for each CB code and provider combination
residential_data.sort_values(by=['CB_Code', 'holding_company', 'max_tech'], ascending=[True, True, False], inplace=True)

### Dropping duplicates to get unique CB code and provider combinations
residential_data.drop_duplicates(subset=['CB_Code', 'holding_company'], inplace=True)

### Resetting index after dropping duplicates
residential_data.reset_index(drop=True, inplace=True)

### Dropping technology columns
residential_data.drop(columns=['technology'], inplace=True)

In [14]:
### Creating categorical column for technology types
def categorize_technology(tech):
    if tech == 50:
        return 'Fiber'
    elif tech == 40:
        return 'Cable'
    else:
        return 'DSL'
residential_data['technology_category'] = residential_data['max_tech'].apply(categorize_technology)

In [15]:
### Checking unique providers
print(residential_data['holding_company'].value_counts())

holding_company
Charter Communications                       159977
Frontier Communications Corporation           80014
Cox Communications, Inc.                      22692
Fidium Holdings, LLC                           7142
Altice USA                                     4370
ConnectTo Communications Inc.                  1662
Wyyerd Group LLC                               1501
Google LLC                                      633
Volcano Communications Company                  555
Sail Internet Holdings LLC                      368
Broadband MDU Holdings, LLC                     262
Atherton Fiber LLC                              221
Hankins Information Technology                  185
Sierra Nevada Communications, LLC               159
Inyo Networks Inc.                              146
Succeed.Net                                      94
Consolidated Smart Broadband Systems, LLC        91
Orion CableSystems, Inc.                         91
Golden Rain Foundation                          

In [16]:
### Creating a categorical column for provider types
def categorize_provider(provider):
    for p in ['AT&T', 'Verizon', 'Cox', 'Comcast', 'Charter', 'CenturyLink', 'Frontier', 'Xfinity', 'Windstream']:
        if p in provider:
            return p
    return 'Other'

### Returning 1 for overbuilders and 0 for non-overbuilders
def overbuilders(provider):
    for p in ['Metro', 'Sparklight', 'T-Mobile', 'Dish', 'Lumen', 'Altice']:
        if p in provider:
            return 1
    return 0

residential_data['provider_category'] = residential_data['holding_company'].apply(categorize_provider)
residential_data['overbuilder_category'] = residential_data['holding_company'].apply(overbuilders)

In [17]:
### Sanity Checks
print(residential_data['provider_category'].value_counts())
residential_data['overbuilder_category'].value_counts()

provider_category
Charter     159977
Frontier     80014
Cox          22692
Other        17713
Name: count, dtype: int64


overbuilder_category
0    276026
1      4370
Name: count, dtype: int64

In [18]:
### For each CB code, creating a column for each provider category and filling it with the technology category for that provider if it exists, otherwise filling it with 'No Service'
provider_categories = residential_data['provider_category'].unique()
for provider in provider_categories:
    residential_data[provider] = residential_data.apply(lambda row: row['technology_category'] if row['provider_category'] == provider else 'No Service', axis=1)

In [ ]:
### For each CB code, adding number of overbuilders providing service in that CB code
overbuilder_counts = residential_data.groupby('CB_Code')['overbuilder_category'].sum().reset_index()
residential_data = pd.merge(residential_data, overbuilder_counts, on='CB_Code', how='left')
residential_data.rename(columns={'overbuilder_category_y': 'overbuilder_count'}, inplace=True)

In [ ]:
### For each CB code, adding major provider count providing service in that CB code
major_provider_counts = residential_data.groupby('CB_Code')['provider_category'].apply(lambda x: (x != 'Other').sum()).reset_index()
residential_data = pd.merge(residential_data, major_provider_counts, on='CB_Code', how='left')
residential_data.rename(columns={'provider_category_y': 'major_provider_count'}, inplace=True)

### For each CB code, adding total provider count providing service in that CB code
total_provider_counts = residential_data.groupby('CB_Code')['provider_category'].apply(lambda x: (x != 'No Service').sum()).reset_index()
residential_data = pd.merge(residential_data, total_provider_counts, on='CB_Code', how='left')
residential_data.rename(columns={'provider_category_y': 'total_provider_count'}, inplace=True)

In [ ]:
### For each CB code, add

In [19]:
residential_data.head()

,provider_id,max_advertised_download_speed,max_advertised_upload_speed,state_usps,holding_company,CB_Code,max_tech,technology_category,provider_category,overbuilder_category_x,Other,Frontier,Charter,Cox,overbuilder_count
0,340066,1000,1000,CA,Sail Internet Holdings LLC,060014007004016,10,DSL,Other,0,DSL,No Service,No Service,No Service,0
1,340066,1000,1000,CA,Sail Internet Holdings LLC,060014008001005,50,Fiber,Other,0,Fiber,No Service,No Service,No Service,0
2,340066,1000,1000,CA,Sail Internet Holdings LLC,060014008001006,50,Fiber,Other,0,Fiber,No Service,No Service,No Service,0
3,340066,1000,1000,CA,Sail Internet Holdings LLC,060014008001007,50,Fiber,Other,0,Fiber,No Service,No Service,No Service,0
4,340066,1000,1000,CA,Sail Internet Holdings LLC,060014008001017,50,Fiber,Other,0,Fiber,No Service,No Service,No Service,0


In [18]:
### Dropping unnecessary columns
columns_to_drop = ['holding_company', 'max_tech', 'technology_category', 'provider_category', 'overbuilder_category_x', 'provider_id']
residential_data.drop(columns=columns_to_drop, inplace=True)

### Reordering columns to have CB code at the front
cols = residential_data.columns.tolist()
cols = cols[-1:] + cols[:-1]
residential_data = residential_data[cols]

### Bringing everything to CB level
final_residential_data = residential_data.groupby('CB_Code').first().reset_index()

In [19]:
### Sanity check
print(final_residential_data.shape)
final_residential_data.head()

(212659, 10)


,CB_Code,Altice USA,provider_id,max_advertised_download_speed,max_advertised_upload_speed,state_usps,Other,Frontier,Charter,Cox
0,060014007004016,No Service,340066,1000,1000,CA,DSL,No Service,No Service,No Service
1,060014008001005,No Service,340066,1000,1000,CA,Fiber,No Service,No Service,No Service
2,060014008001006,No Service,340066,1000,1000,CA,Fiber,No Service,No Service,No Service
3,060014008001007,No Service,340066,1000,1000,CA,Fiber,No Service,No Service,No Service
4,060014008001017,No Service,340066,1000,1000,CA,Fiber,No Service,No Service,No Service
